# 05.2 Capstone: Text Classification / 综合项目：文本分类

这一份 notebook 做一个离线可运行的文本分类综合项目。  
This notebook builds an offline-runnable text classification capstone project.

为了让它在当前环境里稳定可跑，这里使用一个可控的合成情感数据集。  
To keep it stable and runnable in the current environment, we use a controlled synthetic sentiment dataset.

这个项目的重点不是数据来源，而是完整流程 / The focus of this project is not the data source, but the end-to-end workflow:

- 文本预处理 / text preprocessing
- baseline / 基线模型
- 序列模型 / sequence model
- 对照实验 / controlled experiments
- 结果分析 / result analysis

## 学习目标 / Learning Goals

学完后你应该能 / After this notebook, you should be able to:

1. 跑通一个完整的文本分类项目 / Run a complete text classification project.
2. 对比 `bag-of-words / 词袋模型` baseline 和 `LSTM / 长短期记忆网络` / Compare a `bag-of-words` baseline with an `LSTM`.
3. 理解 `tokenization / 分词`、`vocabulary / 词表`、`padding / 补齐` 的项目用法 / Understand the project-level use of tokenization, vocabulary, and padding.
4. 组织训练、验证、测试流程 / Organize train, validation, and test workflows.
5. 分析 negation / 否定词 等顺序敏感现象 / Analyze order-sensitive phenomena such as negation.
6. 把实验结果写成一份清晰总结 / Turn experiment results into a clear summary.

In [ ]:
import copy
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(42)

## 1. 生成一个顺序敏感的数据集 / Generate an Order-Sensitive Dataset

这里的数据被故意设计成包含一些 `negation / 否定` 模式，例如：  
The dataset here is intentionally designed to contain some `negation` patterns, such as:

- `not bad movie` -> positive
- `not good movie` -> negative

这样做的目的，是让顺序信息变得重要。  
The purpose is to make word order matter.

In [ ]:
nouns = ["movie", "film", "story", "show", "plot", "episode"]
positive_templates = [
    ["good", "{noun}"],
    ["really", "good", "{noun}"],
    ["very", "fun", "{noun}"],
    ["not", "bad", "{noun}"],
    ["not", "boring", "{noun}"],
    ["quite", "nice", "{noun}"],
    ["love", "this", "{noun}"],
    ["enjoyable", "{noun}"],
]
negative_templates = [
    ["bad", "{noun}"],
    ["really", "bad", "{noun}"],
    ["very", "boring", "{noun}"],
    ["not", "good", "{noun}"],
    ["not", "fun", "{noun}"],
    ["quite", "weak", "{noun}"],
    ["hate", "this", "{noun}"],
    ["awful", "{noun}"],
]
optional_prefixes = [[], ["overall"], ["today"], ["honestly"], ["for", "me"]]
optional_suffixes = [[], ["overall"], ["for", "me"], ["today"]]


def render_template(template, noun):
    return [token.format(noun=noun) for token in template]


def make_dataset(n_per_label=700):
    texts = []
    labels = []

    for _ in range(n_per_label):
        noun = random.choice(nouns)
        prefix = random.choice(optional_prefixes)
        suffix = random.choice(optional_suffixes)
        tokens = prefix + render_template(random.choice(positive_templates), noun) + suffix
        texts.append(" ".join(tokens))
        labels.append(1)

    for _ in range(n_per_label):
        noun = random.choice(nouns)
        prefix = random.choice(optional_prefixes)
        suffix = random.choice(optional_suffixes)
        tokens = prefix + render_template(random.choice(negative_templates), noun) + suffix
        texts.append(" ".join(tokens))
        labels.append(0)

    combined = list(zip(texts, labels))
    random.shuffle(combined)
    texts, labels = zip(*combined)
    return list(texts), list(labels)


texts, labels = make_dataset(n_per_label=700)
print("dataset size =", len(texts))
print("positive rate / 正样本比例 =", np.mean(labels))
print("sample texts =")
for text, label in list(zip(texts, labels))[:6]:
    print(label, "|", text)

## 2. 训练 / 验证 / 测试拆分
## Train / Validation / Test Split

项目里一个重要习惯是：不要只做 train/test。  
An important project habit is: do not only do train/test.

验证集 / validation set 用来比较方案，测试集 / test set 用来做最终报告。  
The validation set is used to compare ideas; the test set is used for final reporting.

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full,
)

print("train size =", len(X_train))
print("val size =", len(X_val))
print("test size =", len(X_test))

## 3. Baseline 1: Unigram Bag-of-Words / 基线 1：一元词袋模型

第一条 baseline 路线是：  
The first baseline route is:

- `CountVectorizer(ngram_range=(1, 1))`
- `LogisticRegression`

它忽略顺序，只看词出现了多少次。  
It ignores order and only looks at word counts.

In [ ]:
unigram_vectorizer = CountVectorizer(ngram_range=(1, 1))
X_train_uni = unigram_vectorizer.fit_transform(X_train)
X_val_uni = unigram_vectorizer.transform(X_val)
X_test_uni = unigram_vectorizer.transform(X_test)

unigram_lr = LogisticRegression(max_iter=2000, random_state=42)
unigram_lr.fit(X_train_uni, y_train)

unigram_val_preds = unigram_lr.predict(X_val_uni)
unigram_test_preds = unigram_lr.predict(X_test_uni)

unigram_val_acc = accuracy_score(y_val, unigram_val_preds)
unigram_test_acc = accuracy_score(y_test, unigram_test_preds)

print("unigram val acc =", round(unigram_val_acc, 4))
print("unigram test acc =", round(unigram_test_acc, 4))

## 4. Baseline 2: Bigram Bag-of-Words / 基线 2：二元词袋模型

第二条 baseline 是加上 `bigram / 二元词组`。  
The second baseline adds `bigrams`.

这通常能更好地处理像 `not good` 这样的局部顺序模式。  
This can usually handle local order patterns such as `not good` better.

In [ ]:
bigram_vectorizer = CountVectorizer(ngram_range=(1, 2))
X_train_bi = bigram_vectorizer.fit_transform(X_train)
X_val_bi = bigram_vectorizer.transform(X_val)
X_test_bi = bigram_vectorizer.transform(X_test)

bigram_lr = LogisticRegression(max_iter=2000, random_state=42)
bigram_lr.fit(X_train_bi, y_train)

bigram_val_preds = bigram_lr.predict(X_val_bi)
bigram_test_preds = bigram_lr.predict(X_test_bi)

bigram_val_acc = accuracy_score(y_val, bigram_val_preds)
bigram_test_acc = accuracy_score(y_test, bigram_test_preds)

print("bigram val acc =", round(bigram_val_acc, 4))
print("bigram test acc =", round(bigram_test_acc, 4))

## 5. 为 LSTM 准备数据 / Prepare Data for the LSTM

现在换成 `PyTorch` 的序列建模路线。  
Now we switch to the `PyTorch` sequence-modeling route.

这里需要三步 / We need three steps here:

1. `tokenization / 分词`
2. `vocabulary / 词表`
3. `padding / 补齐`

In [ ]:
def tokenize(text):
    return text.split()


special_tokens = ["<pad>", "<unk>"]
vocab = sorted({token for text in X_train for token in tokenize(text)})
vocab = special_tokens + vocab
stoi = {token: idx for idx, token in enumerate(vocab)}
pad_id = stoi["<pad>"]
unk_id = stoi["<unk>"]
max_len = max(len(tokenize(text)) for text in X_train)

print("vocab size =", len(vocab))
print("max_len =", max_len)
print("first vocab items =", vocab[:12])

In [ ]:
def encode_text(text, max_len=max_len):
    tokens = tokenize(text)
    ids = [stoi.get(token, unk_id) for token in tokens][:max_len]
    while len(ids) < max_len:
        ids.append(pad_id)
    return ids


def build_tensor_dataset(texts, labels):
    x = torch.tensor([encode_text(text) for text in texts], dtype=torch.long)
    y = torch.tensor(labels, dtype=torch.long)
    return TensorDataset(x, y)


train_ds = build_tensor_dataset(X_train, y_train)
val_ds = build_tensor_dataset(X_val, y_val)
test_ds = build_tensor_dataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

xb, yb = next(iter(train_loader))
print("xb.shape =", xb.shape)
print("yb.shape =", yb.shape)
print("sample encoded =", xb[0])

## 6. LSTM 模型与训练函数 / LSTM Model and Training Function

`LSTM / 长短期记忆网络` 会按序列顺序读 token。  
An `LSTM` reads tokens in sequence order.

因此它理论上比 unigram 词袋更适合处理顺序敏感模式。  
So in principle it is better suited than unigram bag-of-words for order-sensitive patterns.

In [ ]:
class TextLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=48, pad_id=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        emb = self.embedding(x)
        _, (h_n, _) = self.lstm(emb)
        last_hidden = h_n[-1]
        return self.fc(last_hidden)


def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    total_correct = 0
    total_items = 0

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            preds = logits.argmax(dim=1)
            total_loss += loss.item() * xb.size(0)
            total_correct += (preds == yb).sum().item()
            total_items += xb.size(0)

    return total_loss / total_items, total_correct / total_items


def train_lstm(config):
    set_seed(42)
    model = TextLSTMClassifier(
        vocab_size=len(vocab),
        embed_dim=config["embed_dim"],
        hidden_dim=config["hidden_dim"],
        pad_id=pad_id,
    )
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])

    history = []
    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = -1.0

    for epoch in range(1, config["epochs"] + 1):
        train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
            }
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


def collect_predictions(model, loader):
    model.eval()
    preds = []
    targets = []
    with torch.no_grad():
        for xb, yb in loader:
            preds.append(model(xb).argmax(dim=1))
            targets.append(yb)
    return torch.cat(preds), torch.cat(targets)

## 7. 训练 LSTM / Train the LSTM

这里把 `LSTM` 作为顺序模型路线。  
Here we use the `LSTM` as the sequence-model route.

In [ ]:
lstm_config = {
    "embed_dim": 32,
    "hidden_dim": 48,
    "lr": 0.01,
    "epochs": 10,
}

lstm_model, lstm_history = train_lstm(lstm_config)
lstm_val_preds, lstm_val_targets = collect_predictions(lstm_model, val_loader)
lstm_test_preds, lstm_test_targets = collect_predictions(lstm_model, test_loader)

lstm_val_acc = accuracy_score(lstm_val_targets.numpy(), lstm_val_preds.numpy())
lstm_test_acc = accuracy_score(lstm_test_targets.numpy(), lstm_test_preds.numpy())

print("lstm val acc =", round(lstm_val_acc, 4))
print("lstm test acc =", round(lstm_test_acc, 4))

## 8. 结果表 / Result Table

现在把三条路线放在同一张表里比较：  
Now we compare the three routes in one table:

- unigram bag-of-words
- bigram bag-of-words
- LSTM

In [ ]:
results_df = pd.DataFrame(
    [
        {
            "model": "Unigram-LR",
            "family": "baseline",
            "val_acc": round(unigram_val_acc, 4),
            "test_acc": round(unigram_test_acc, 4),
            "notes": "bag-of-words without order",
        },
        {
            "model": "Bigram-LR",
            "family": "baseline",
            "val_acc": round(bigram_val_acc, 4),
            "test_acc": round(bigram_test_acc, 4),
            "notes": "bag-of-words with local order patterns",
        },
        {
            "model": "LSTM",
            "family": "torch",
            "val_acc": round(lstm_val_acc, 4),
            "test_acc": round(lstm_test_acc, 4),
            "notes": "sequence model with recurrent order modeling",
        },
    ]
).sort_values(by=["test_acc", "val_acc"], ascending=False)
results_df

## 9. LSTM 训练曲线 / LSTM Training Curves

这里只画 `LSTM` 的训练曲线，因为两个 `LogisticRegression` baseline 没有 epoch history。  
We only plot the `LSTM` training curves because the two `LogisticRegression` baselines do not have epoch histories.

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(lstm_history["epoch"], lstm_history["train_loss"], label="train loss")
plt.plot(lstm_history["epoch"], lstm_history["val_loss"], label="val loss")
plt.title("LSTM Loss Curves")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(lstm_history["epoch"], lstm_history["train_acc"], label="train acc")
plt.plot(lstm_history["epoch"], lstm_history["val_acc"], label="val acc")
plt.title("LSTM Accuracy Curves")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()

plt.tight_layout()
plt.show()
plt.close()

## 10. 选择最优模型 / Select the Best Model

这里依然按 `test_acc` 最高来选最终模型，仅用于教学展示。  
Here we again select the final model by the highest `test_acc`, only for teaching demonstration.

In [ ]:
all_candidates = {
    "Unigram-LR": {
        "preds": np.array(unigram_test_preds),
        "targets": np.array(y_test),
        "test_acc": unigram_test_acc,
    },
    "Bigram-LR": {
        "preds": np.array(bigram_test_preds),
        "targets": np.array(y_test),
        "test_acc": bigram_test_acc,
    },
    "LSTM": {
        "preds": lstm_test_preds.numpy(),
        "targets": lstm_test_targets.numpy(),
        "test_acc": lstm_test_acc,
    },
}

best_name = max(all_candidates, key=lambda name: all_candidates[name]["test_acc"])
best_preds = all_candidates[best_name]["preds"]
best_targets = all_candidates[best_name]["targets"]

print("best model =", best_name)
print("best test acc =", round(all_candidates[best_name]["test_acc"], 4))

## 11. Confusion Matrix / 混淆矩阵

这里是二分类，所以矩阵很小，但仍然有价值。  
This is binary classification, so the matrix is small, but it is still useful.

它可以帮助你看出：模型是更容易错判正类，还是更容易错判负类。  
It helps you see whether the model is more likely to miss positives or miss negatives.

In [ ]:
cm = confusion_matrix(best_targets, best_preds)
plt.figure(figsize=(4, 4))
plt.imshow(cm, cmap="Blues")
plt.title(f"Confusion Matrix: {best_name}")
plt.xlabel("predicted label")
plt.ylabel("true label")
plt.colorbar()

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

plt.tight_layout()
plt.show()
plt.close()

print(classification_report(best_targets, best_preds, digits=4, target_names=["negative", "positive"]))

## 12. 误分类文本分析 / Misclassified Text Analysis

文本任务里，误分类样本往往比数字本身更有解释性。  
In text tasks, the misclassified examples are often more interpretable than the metric itself.

In [ ]:
mis_positions = np.where(best_preds != best_targets)[0]
mis_rows = []
for pos in mis_positions[:12]:
    mis_rows.append(
        {
            "text": X_test[pos],
            "true_label": int(best_targets[pos]),
            "pred_label": int(best_preds[pos]),
        }
    )

mis_df = pd.DataFrame(mis_rows)
print("num misclassified =", len(mis_positions))
mis_df

## 13. 结果解读 / Interpreting the Results

这个项目的一个关键问题是：  
One key question in this project is:

- 局部顺序模式（例如 `not good`）到底需要多强的建模能力？ / how much modeling power is needed for local order patterns such as `not good`?

如果 `Bigram-LR` 已经很强，说明很多信息可以被局部 n-gram 捕获。  
If `Bigram-LR` is already very strong, it means much of the information can be captured by local n-grams.

如果 `LSTM` 进一步提升，说明更完整的序列建模确实有帮助。  
If the `LSTM` further improves performance, it suggests that fuller sequence modeling is indeed helpful.

In [ ]:
summary_lines = [
    f"Best model: {best_name}",
    f"Unigram-LR test accuracy: {unigram_test_acc:.4f}",
    f"Bigram-LR test accuracy: {bigram_test_acc:.4f}",
    f"LSTM test accuracy: {lstm_test_acc:.4f}",
    f"Number of misclassified test samples: {len(mis_positions)}",
]

for line in summary_lines:
    print(line)

In [ ]:
# 练习 1 / Exercise 1
# 为什么 unigram 词袋模型在处理 'not good' 和 'good' 时可能吃亏？
# Why can a unigram bag-of-words model struggle with 'not good' versus 'good'?

练习 1 参考答案 / Exercise 1 Reference Answer

因为 unigram 只看单个词，不直接表示相邻词之间的顺序关系。  
Because a unigram model only looks at individual words and does not directly represent the order relation between adjacent words.

它知道 `not` 出现了，也知道 `good` 出现了，但不一定知道它们组合成了 `not good`。  
It knows that `not` appears and that `good` appears, but not necessarily that they form the phrase `not good`.

In [ ]:
# 练习 2 / Exercise 2
# 如果 Bigram-LR 已经和 LSTM 很接近，这说明了什么？
# If Bigram-LR is already very close to the LSTM, what does that suggest?

练习 2 参考答案 / Exercise 2 Reference Answer

这通常说明当前任务里，关键信息大多可以被局部短语模式捕获。  
This usually suggests that in the current task, most of the crucial information can be captured by local phrase patterns.

也就是说，序列模型未必总能显著胜出，是否值得更复杂建模要看任务本身。  
In other words, a sequence model does not always win by a large margin; whether the extra complexity is worth it depends on the task.

## 14. 小结 / Summary

这一份文本综合项目里，你已经练习了完整链条：  
In this text capstone, you have practiced the full chain:

1. 构造任务 / task construction
2. 文本预处理 / text preprocessing
3. baseline 对比 / baseline comparison
4. `LSTM` 序列建模 / `LSTM` sequence modeling
5. 结果表、训练曲线、混淆矩阵、误分类分析 / result tables, training curves, confusion matrices, and misclassification analysis
6. 实验结论总结 / experiment conclusion writing